# EVE 310 - Lab 10: Clustering with k-means

**Module 4 | 11/05/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab10-clustering/notebooks/lab10-tutorial.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Fit a k-means model with scikit-learn and read its two results: a cluster label for every point and a centroid for every cluster
2. Measure a clustering with the SSE (`inertia_`) and the silhouette score, and say what each one rewards
3. Choose the number of clusters with a for loop over k: the elbow curve and the silhouette curve
4. Explain why the columns must be on the same scale before k-means (the activity)

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It downloads the two data files into `DATA_DIR` and creates `FIGURES_DIR`.
3. **This tutorial is typed, not read.** Most code cells are missing one or two lines, marked `--`. In lab, type them along with the instructor, then run the cell. The text above each cell says what the lines do, and **Show the finished cell** below it has the answer if you get stuck or are working on your own. A cell with a `--` still in it cannot run, so **Run all** fails until every blank is filled.
4. Nothing to submit for this notebook. The **activity** notebook is the one you download as `.ipynb` and upload to Gradescope (optional, for feedback).

### Tutorial

Every model so far had an answer column. Lab 5 fit a household's daily trips to the number of people in it, Lab 8 turned a model's probabilities into yes/no detections for the TCE wells: in each case the data came with a `y`, and the model was judged by how well it reproduced it. That is **supervised** learning. Today there is no `y`. The data is a table of measurements and nothing else, and the question is whether the rows fall into **groups**: buildings that use water the same way, wells with the same chemistry, households with the same habits. Finding those groups is **clustering**, the first **unsupervised** method of the course, and k-means is the standard way to do it.

You met clustering and k-means in lecture. This notebook defines everything it uses, so treat it as a first pass, not a review. The Python is small: one new scikit-learn class, `KMeans`, whose `fit` and `predict` work like `LinearRegression` did in Labs 5 and 6, minus one argument, and one new scoring function. The loops are Lab 8's sweep: a grid of values, an empty array, a result stored per pass.

**How k-means works, in one paragraph.** You choose the number of clusters, `k`. The algorithm places `k` **centroids** (one point per cluster, the cluster's center), then repeats two steps: assign every point to its nearest centroid, and move each centroid to the mean of the points assigned to it. When the assignments stop changing it is done. The result is a **label** for every point (which cluster, numbered 0 to k - 1) and the final centroids. The quality of a result is its **SSE**, the sum over all points of the squared distance from the point to its own centroid: the smaller, the tighter the clusters. The cluster numbers themselves mean nothing: cluster 0 in one run can be cluster 2 in a run started from a different `random_state`, with not one point moved. What matters is which points share a number.

The example is the lecture's: 19 points with two measurements each, `x1` and `x2`. Small enough to see every point, and big enough to show the three ideas of the lab: fit, measure, choose k. The activity then does the same on 300 households, where the columns are in different units and one more step becomes essential.

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab10-clustering"
DATA_FILES = ["lecture_points.csv", "household_water.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**Run as is:** the usual three, plus the two scikit-learn names of the day. `KMeans` is the model, from `sklearn.cluster`. `silhouette_score` is a scoring function, from `sklearn.metrics`, the module `precision_score` came from in Lab 8.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1. The data: no answer column

`lecture_points.csv` has 19 rows and two columns, `x1` and `x2`. Nothing else: no label, no category, no `y`. The scatter plot below is all the information there is, and the question is simply whether you see groups.

**Type it:** two lines. Read the file into `points_df` with `pd.read_csv(DATA_DIR / 'lecture_points.csv')`, then build `X`, the two columns as one NumPy array: `X = points_df[['x1', 'x2']].to_numpy()`. Two pairs of brackets: the inner list names the columns, the outer brackets select them (the double brackets of Lab 6's `describe()` table), and `.to_numpy()`, which Labs 8 and 9 used on one column, turns the selection into an array with one row per point. scikit-learn wants its inputs this way, one row per observation and one column per measurement, as the regression `x` was in Labs 5 and 6.

In [ ]:
# Read the 19 points, then put the two columns in one array, one row per point
--
--

print(X.shape, '= (points, measurements)')
print(X[:5])

fig, ax = plt.subplots(figsize=(5, 5))
# X[:, 0] is column 0 (x1) of every row, X[:, 1] is column 1 (x2)
ax.scatter(X[:, 0], X[:, 1], s=80, color='k')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('19 points, no labels')

<details><summary>Show the finished cell</summary>

```python
# Read the 19 points, then put the two columns in one array, one row per point
points_df = pd.read_csv(DATA_DIR / 'lecture_points.csv')
X = points_df[['x1', 'x2']].to_numpy()

print(X.shape, '= (points, measurements)')
print(X[:5])

fig, ax = plt.subplots(figsize=(5, 5))
# X[:, 0] is column 0 (x1) of every row, X[:, 1] is column 1 (x2)
ax.scatter(X[:, 0], X[:, 1], s=80, color='k')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('19 points, no labels')
```

</details>

`X` has 19 rows and 2 columns. `X[:, 0]` is the first column (every row, column 0) and `X[:, 1]` the second, which is how the scatter plot pulls out `x1` and `x2`. Most people see three groups: left, upper middle, lower right. k-means has to find them from the coordinates alone.

## 2. Fit k-means with k = 3

The model is built the way every scikit-learn model has been built: create it, then `fit` it. Three arguments:

- `n_clusters=k`: how many clusters to look for. k-means never chooses this for you; section 5 shows how to choose it.
- `random_state=0`: the starting centroids are chosen at random, so this fixes the draw, as `np.random.seed(0)` did in Lab 7. Everyone in the room gets the same result.
- `n_init=10`: run the whole algorithm from 10 different random starts and keep the run with the smallest SSE. A single start can land in a poor local arrangement; ten starts almost never do.

`fit` takes only `X`. There is no `y` to pass, and that one missing argument is the whole difference between supervised and unsupervised learning.

**Type it:** `kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)`, then `kmeans.fit(X)`.

In [ ]:
k = 3

# Create the model with k clusters, then fit it to X (no y: there is none)
--
--

print(kmeans)

<details><summary>Show the finished cell</summary>

```python
k = 3

# Create the model with k clusters, then fit it to X (no y: there is none)
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
kmeans.fit(X)

print(kmeans)
```

</details>

## 3. The two results: labels and centroids

A fitted k-means model holds two things, both with the trailing underscore scikit-learn puts on everything learned from data (as in `coef_` and `intercept_`):

- `kmeans.labels_`: an array with one whole number per row of `X`, the cluster that row was assigned to, 0 to k - 1. Same length as `X`, same order.
- `kmeans.cluster_centers_`: a k by 2 array, one row per cluster, the final centroid in the units of `x1` and `x2`.

The plot colors each point by its label (`c=labels` gives `scatter` one number per point and lets it pick the colors) and draws the centroids as red crosses.

**Type it:** `labels = kmeans.labels_` and `centroids = kmeans.cluster_centers_`.

In [ ]:
# The cluster of every point, and the center of every cluster
--
--

print('labels:   ', labels)
print('centroids:')
print(centroids)
# np.bincount counts how many points carry each label: the size of each cluster
print('points per cluster:', np.bincount(labels))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], c=labels, s=80)
ax.scatter(centroids[:, 0], centroids[:, 1], marker='x', s=250, color='red', linewidths=3)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('k-means, k = 3')

<details><summary>Show the finished cell</summary>

```python
# The cluster of every point, and the center of every cluster
labels = kmeans.labels_
centroids = kmeans.cluster_centers_

print('labels:   ', labels)
print('centroids:')
print(centroids)
# np.bincount counts how many points carry each label: the size of each cluster
print('points per cluster:', np.bincount(labels))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], c=labels, s=80)
ax.scatter(centroids[:, 0], centroids[:, 1], marker='x', s=250, color='red', linewidths=3)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('k-means, k = 3')
```

</details>

Seven points in one cluster, six in each of the other two, and the three crosses sit in the middle of the three groups you saw by eye. `np.bincount(labels)` counts how many times each label appears, which is the size of each cluster. The labels print as 0s, 1s and 2s in the row order of `X`: point 0 and point 1 share a number, so they are in the same cluster. Which number that is carries no meaning.

## 4. Two ways to score a clustering

**SSE.** k-means minimizes one quantity: the sum, over all points, of the squared distance from the point to its own centroid. scikit-learn calls it `inertia_`; the lecture calls it SSE, the same name as Lab 5's sum of squared errors, and for the same reason: it is a sum of squared errors, where the error is now the gap between a point and its cluster center. Smaller is tighter.

The second line of the cell computes the same number by hand, so you can see what it is. `centroids[labels]` uses the labels array as row indices into `centroids`, which lines up each point's own centroid next to it (19 rows, like `X`). Subtract, square, add everything up.

**Silhouette score.** SSE has a flaw you will see in section 5: it keeps falling as k grows, so it cannot say on its own which k is best. The **silhouette score** compares, for each point, how close it is to its own cluster against how close it is to the nearest other cluster. It runs from -1 to 1: near 1 means the point sits deep inside its own cluster, near 0 means it is on a border between two clusters, below 0 means it probably belongs to the other cluster. `silhouette_score(X, labels)` averages it over every point, so one number describes the whole clustering, and it takes the points and the labels, not the model.

**Type it:** `sse = kmeans.inertia_`, and in the second cell `sc = silhouette_score(X, labels)`.

In [ ]:
# The model's own score: the sum of squared distances from each point to its centroid
--
# The same number by hand: each point's own centroid, lined up with centroids[labels]
sse_by_hand = np.sum((X - centroids[labels]) ** 2)

print('SSE from the model:', round(sse, 2))
print('SSE by hand:       ', round(sse_by_hand, 2))

<details><summary>Show the finished cell</summary>

```python
# The model's own score: the sum of squared distances from each point to its centroid
sse = kmeans.inertia_
# The same number by hand: each point's own centroid, lined up with centroids[labels]
sse_by_hand = np.sum((X - centroids[labels]) ** 2)

print('SSE from the model:', round(sse, 2))
print('SSE by hand:       ', round(sse_by_hand, 2))
```

</details>

In [ ]:
# The silhouette score: how well separated the clusters are, from -1 to 1
--

print('silhouette score:', round(sc, 3))

<details><summary>Show the finished cell</summary>

```python
# The silhouette score: how well separated the clusters are, from -1 to 1
sc = silhouette_score(X, labels)

print('silhouette score:', round(sc, 3))
```

</details>

Both SSE lines print 1624.43, which is the whole definition of `inertia_` in one line of NumPy. The silhouette score of 0.64 says the three groups are well separated: most points are much closer to their own centroid than to any other. Neither number means much on its own. They are for **comparing** clusterings, which is the next section.

## 5. Choosing k: the elbow curve

k was 3 because the plot looked like three groups. With more than two columns there is no plot to look at, and even with two (the activity) the eye is a guess, not a rule, so k has to be chosen from numbers, and the only way is to try several values. That is Lab 8's sweep: a grid of k values, an empty array with one slot per k, and a loop that fits one model per pass and stores its SSE in slot `i`.

The model inside the loop is named `kmeans_k`, not `kmeans`, so that your k = 3 model from section 2 survives the loop (Lab 9: a variable assigned inside a loop holds only the last pass). `.fit(X)` is chained straight onto `KMeans(...)`, which creates and fits in one line; `fit` returns the fitted model, so `kmeans_k` holds it.

**Type it:** the two lines inside the loop: `kmeans_k = KMeans(n_clusters=k_array[i], random_state=0, n_init=10).fit(X)`, then `sse_array[i] = kmeans_k.inertia_`. The current k is `k_array[i]`, not `k`.

In [ ]:
k_array = np.arange(1, 9)
sse_array = np.zeros(len(k_array))

for i in range(len(k_array)):
    # Fit a model with the current k, then store its SSE in slot i
    --
    --

print(np.round(sse_array, 1))

fig, ax = plt.subplots()
ax.plot(k_array, sse_array, 'o-')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('SSE')
ax.set_title('The elbow curve')

<details><summary>Show the finished cell</summary>

```python
k_array = np.arange(1, 9)
sse_array = np.zeros(len(k_array))

for i in range(len(k_array)):
    # Fit a model with the current k, then store its SSE in slot i
    kmeans_k = KMeans(n_clusters=k_array[i], random_state=0, n_init=10).fit(X)
    sse_array[i] = kmeans_k.inertia_

print(np.round(sse_array, 1))

fig, ax = plt.subplots()
ax.plot(k_array, sse_array, 'o-')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('SSE')
ax.set_title('The elbow curve')
```

</details>

SSE falls at every step: 13,654 for one cluster (every point against the overall mean), 5,023 for two, 1,624 for three, and then slowly on to 401 for eight. The **best possible** SSE can never go up with k: keep the previous k centroids and add one, and no point is farther from its centroid than before; at k = 19 every point would be its own centroid with SSE = 0. k-means does not guarantee it finds that best arrangement (it restarts from random centroids at each k), but ten starts get close enough that the curve falls at every step here, as it does in the activity. So "smallest SSE" is useless as a rule for choosing k. The rule is the **elbow**: the k where the curve stops dropping steeply and starts to flatten. Here the big drops end at k = 3; from 3 on, each extra cluster buys little. An elbow is a judgment, not a formula, which is why a second opinion helps.

## 6. Your turn: the silhouette curve

Same loop, other score. For k from 2 to 8 (the silhouette needs at least two clusters), fit the model, store the silhouette score of its labels in `sc_array[i]`, and after the loop take the k with the **largest** score. The fit line is written for you; it is section 5's. Two `--` to fill: the silhouette line is section 4's with the current model's labels, `kmeans_k.labels_`, in place of `labels`. The `best_k` line is new today: `np.argmax(sc_array)` gives the **slot** (position) of the largest value, one call in place of Lab 7's largest-so-far loop, and `k_array[that slot]` turns the position back into a k, the way `threshold_array[i]` turned slot `i` into a threshold in Lab 8.

In [ ]:
k_array = np.arange(2, 9)
sc_array = np.zeros(len(k_array))

for i in range(len(k_array)):
    kmeans_k = KMeans(n_clusters=k_array[i], random_state=0, n_init=10).fit(X)
    # Store the silhouette score of this model's labels in slot i
    sc_array[i] = --

# The k whose score is largest: the slot of the largest value, turned back into a k
best_k = --

print(np.round(sc_array, 3))
print('best k by silhouette:', best_k)

fig, ax = plt.subplots()
ax.plot(k_array, sc_array, 'o-')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('silhouette score')
ax.set_title('The silhouette curve')

<details><summary>Show the finished cell</summary>

```python
k_array = np.arange(2, 9)
sc_array = np.zeros(len(k_array))

for i in range(len(k_array)):
    kmeans_k = KMeans(n_clusters=k_array[i], random_state=0, n_init=10).fit(X)
    # Store the silhouette score of this model's labels in slot i
    sc_array[i] = silhouette_score(X, kmeans_k.labels_)

# The k whose score is largest: the slot of the largest value, turned back into a k
best_k = k_array[np.argmax(sc_array)]

print(np.round(sc_array, 3))
print('best k by silhouette:', best_k)

fig, ax = plt.subplots()
ax.plot(k_array, sc_array, 'o-')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('silhouette score')
ax.set_title('The silhouette curve')
```

</details>

The silhouette peaks at k = 3 (0.640), where the elbow was, and falls off on both sides: k = 2 lumps two groups together (0.572), k = 4 splits one (0.599), and from k = 5 the clusters are too small to be well separated. When the two curves agree, as here, the choice is easy. When they disagree, the silhouette is usually the better guide, because it rewards separation and not only tightness; and whichever k you choose, say why.

## 7. Using the model: new points

**Run as is.** A fitted model can place a **new** point: `predict` assigns it to the nearest centroid, with no refitting, the way `lm.predict(x_test)` scored the testing rows in Labs 5 and 6. The model is `kmeans`, the k = 3 fit from section 2, which the loops left alone because they used `kmeans_k`. The new points must be a 2-D array with the same two columns as `X`, in the same order and the same units.

In [ ]:
new_points = np.array([[30, 50],
                       [60, 20]])
print('cluster of each new point:', kmeans.predict(new_points))
print('for comparison, the centroids:')
print(np.round(centroids, 1))

Each new point gets the label of the centroid it is closest to, which you can confirm from the centroid table. This is the step a utility runs every month on new customers once the clusters are fixed. It is also where the activity's big caveat bites: a new point in the wrong units lands in the wrong cluster.

**Three things to remember about k-means**

1. **No `y`, so nothing to be right about.** Supervised models are checked against the truth; a clustering is checked against itself (SSE, silhouette) and against whether the groups make sense to the person who asked. Two people can choose different k from the same curves.
2. **The label numbers are arbitrary.** Cluster 0 is whichever cluster the algorithm numbered first. Compare clusterings by which points share a label and by the centroids, never by the numbers. A different `random_state` can renumber the clusters without moving a single point.
3. **k-means measures distance, so the columns must be comparable.** A column in thousands drowns a column in ones: the distance is all the big column. The 19 points had the same units in both columns; the activity's households do not, and standardizing the columns first (Lab 6's z-score) is what makes the clusters meaningful.

**Common mistakes**

| What you see | What went wrong |
| --- | --- |
| `TypeError: KMeans.fit() missing 1 required positional argument: 'X'` | `kmeans.fit()` with nothing inside the parentheses. k-means takes `X`, and only `X` (a `y` passed out of habit is silently ignored) |
| `NameError: name 'y' is not defined` | `kmeans.fit(X, y)` out of habit. There is no `y` today |
| `UserWarning: X does not have valid feature names` when you call `predict` | The model was fitted on the dataframe, `fit(points_df)`, and `predict` then got an array. Fit on `X`, the array, as in section 2 |
| `AttributeError: 'KMeans' object has no attribute 'labels_'` | The model was created but not fitted. `labels_` and `cluster_centers_` exist only after `fit` |
| `labels` are all the same, or `np.bincount` shows one cluster | `n_clusters=1`, or `predict` was run on the wrong array |
| `ValueError: Expected 2D array, got 1D array` | `predict([30, 50])` with one pair of brackets. One new point is still a 2-D array: `[[30, 50]]` |
| `sse_array` is all zeros | The `sse_array[i] = ...` line is missing or not indented into the loop |
| `IndexError: index 8 is out of bounds` | `range(len(k_array))` gives 0 to 7; using `k_array[i]` as a slot index mixes up the k value with its position |
| `kmeans.predict` gives clusters that make no sense after the loops | The loop reused the name `kmeans`, so it now holds the k = 8 model. Name the loop's model `kmeans_k` |
| `ValueError: Number of labels is 1` from `silhouette_score` | The silhouette needs at least two clusters; the k grid must start at 2 |

## 8. Why this matters

- **Customer segmentation for a utility:** group households by how and when they use water or electricity, then design one rebate or one message per group instead of one for everyone. The activity does the two-column version.
- **Finding the odd group:** a cluster nobody expected (a small group with high night-time water flow) is a lead: leaks, a broken meter, an illegal connection. Clustering finds it without anyone having labeled a single leak.
- **Lab 9, again:** the loop over k is the loop over files with a model fitted per pass instead of a file read per pass, and `kmeans_k` versus `kmeans` is the same lesson as `water_df` holding only the last file.
- **Lab 8, again:** `k_array`, `sse_array`, `sc_array` and the loop over `range(len(k_array))`: the sweep is the same code with a new quantity swept. Only the `np.argmax` line at the end is new.

Now open the **activity** notebook.

## Wrap-up

Nothing to submit for the tutorial. Keep it open: the activity uses the same `KMeans`, the same two scores and the same loop over k on 300 households, with one new step first.